# Gaussian Processes — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## A kernel turns nearby inputs into correlated function values
Gaussian Processes put a joint Gaussian distribution over function values. These examples build an RBF kernel, condition on observations, compute posterior uncertainty, vary length-scale, and sample functions.

In [ ]:
# 1) RBF kernel matrix for three inputs
X=np.array([-1.,0.,1.]); ell=1.0; K=np.exp(-0.5*((X[:,None]-X[None,:])**2)/ell**2)
plt.figure(figsize=(6,3)); plt.imshow(K); plt.colorbar(); plt.title('kernel covariance matrix')
assert abs(K[0,2]-0.1353352832366127)<1e-12

In [ ]:
# 2) GP posterior mean on a grid
Xt=np.array([-1.,1.]); y=np.array([-0.8,0.9]); noise=0.1; Xs=np.linspace(-2,2,80)
Ktt=np.exp(-0.5*((Xt[:,None]-Xt[None,:])**2))+noise*np.eye(2); Kst=np.exp(-0.5*((Xs[:,None]-Xt[None,:])**2)); mean=Kst@np.linalg.solve(Ktt,y)
plt.figure(figsize=(6,3)); plt.plot(Xs,mean); plt.scatter(Xt,y,c='r'); plt.title('posterior mean interpolates evidence')
assert mean[0]<0 and mean[-1]>0

In [ ]:
# 3) posterior variance is low near observed inputs
Kss=np.ones(len(Xs)); v=Kss-np.sum(Kst@np.linalg.inv(Ktt)*Kst,axis=1)
plt.figure(figsize=(6,3)); plt.plot(Xs,v); plt.scatter(Xt,[0,0],c='r'); plt.title('uncertainty dips near data')
assert v[len(v)//2]>v[np.argmin(abs(Xs-1))]

In [ ]:
# 4) longer length-scale creates smoother correlation
ells=[0.4,1.0,2.0]; covs=[math.exp(-0.5*(1.0/e)**2) for e in ells]
plt.figure(figsize=(6,3)); plt.plot(ells,covs,'-o'); plt.xlabel('length-scale'); plt.ylabel('corr(distance=1)'); plt.title('length-scale controls smoothness')
assert covs[-1]>covs[0]

In [ ]:
# 5) prior samples are correlated functions
rng=np.random.default_rng(1); Xg=np.linspace(-2,2,40); Kg=np.exp(-0.5*((Xg[:,None]-Xg[None,:])**2))+1e-6*np.eye(40); samples=rng.multivariate_normal(np.zeros(40),Kg,3)
plt.figure(figsize=(6,3)); plt.plot(Xg,samples.T); plt.title('GP prior samples')
assert samples.shape==(3,40)